# Part 2 — Information Extraction Pipeline
## Design Axis: End-to-end vs Decomposed

**Hypothesis:** End-to-end extraction will have higher coverage but lower precision.
Decomposed pipelines will have higher precision because each step is a simpler task.

**Model used:** `google/flan-t5-base` — free, runs on Colab GPU, no API key needed.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2: Auto-download dataset — no manual upload needed
import os
from pathlib import Path

BASE      = Path('/content/ebm_nlp_2_00')
DOCS_DIR  = BASE / 'documents'
ANNOT_DIR = BASE / 'annotations' / 'aggregated' / 'hierarchical_labels'

if not BASE.exists():
    print('Downloading EBM-NLP dataset (~30MB)...')
    os.system('wget -q "https://github.com/bepnye/EBM-NLP/raw/master/ebm_nlp_2_00.tar.gz" -O /content/ebm_nlp_2_00.tar.gz')
    os.system('tar -xzf /content/ebm_nlp_2_00.tar.gz -C /content/')
    print('Download complete.')
else:
    print('Dataset already exists.')

assert DOCS_DIR.exists(),  'Documents folder missing'
assert ANNOT_DIR.exists(), 'Annotations folder missing'
print(f'Ready. Documents: {DOCS_DIR}')

Dataset already exists.
Ready. Documents: /content/ebm_nlp_2_00/documents


In [ ]:
# Installing required libraries
!pip install transformers sentencepiece -q


In [ ]:
# Loading a QA model instead of Flan-T5
from transformers import pipeline
import torch

device = 0 if torch.cuda.is_available() else -1
print(f'Using GPU: {device == 0}')

# This model is trained specifically to extract spans from text
# It answers questions like "who are the patients?" by finding the relevant phrase directly in the abstract
qa_pipeline = pipeline(
    'question-answering',
    model='deepset/roberta-base-squad2',
    device=device
)
print('Model ready.')

Using GPU: True


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready.


In [ ]:
#Testing the model if it is working
test_context = 'Forty-five patients with chronic obstructive pulmonary disease received pulmonary rehabilitation before surgery.'
test_question = 'Who are the patients?'
result = qa_pipeline(question=test_question, context=test_context)
print('Test answer:', result['answer'])
print('Confidence:', round(result['score'], 3))


Test answer: Forty-five patients with chronic obstructive pulmonary disease
Confidence: 0.411


In [ ]:
#Loading a small set of abstracts for extraction
# We use 20 abstracts — enough to show meaningful differences between approaches without taking too long to run
from pathlib import Path

# Already be defined from the clustering notebook.
# If we get a NameError, we re-run Cells 3 to5 of the clustering notebook.
BASE      = Path('/content/ebm_nlp_2_00')
DOCS_DIR  = BASE / 'documents'
ANNOT_DIR = BASE / 'annotations' / 'aggregated' / 'hierarchical_labels'

def get_all_pmids(split='train'):
    ann_folder = ANNOT_DIR / 'participants' / split
    pmids = []
    for f in ann_folder.iterdir():
        if f.suffix == '.ann':
            pmid = f.stem.split('.')[0]
            if (DOCS_DIR / f'{pmid}.tokens').exists():
                pmids.append(pmid)
    return sorted(pmids)

def load_one_abstract(pmid):
    token_file = DOCS_DIR / f'{pmid}.tokens'
    sent_file  = DOCS_DIR / f'{pmid}.sentences'
    if not token_file.exists():
        return None
    tokens = token_file.read_text(encoding='utf-8', errors='replace').strip().split('\n')
    sentences = []
    if sent_file.exists():
        for line in sent_file.read_text().strip().split('\n'):
            parts = line.strip().split()
            if len(parts) == 2:
                sentences.append((int(parts[0]), int(parts[1])))
    else:
        sentences = [(0, len(tokens))]
    return {'pmid': pmid, 'tokens': tokens, 'sentences': sentences}

def load_ann_labels(pmid, field, split='train'):
    ann_file = ANNOT_DIR / field / split / f'{pmid}.AGGREGATED.ann'
    if not ann_file.exists():
        return []
    labels = []
    for line in ann_file.read_text().strip().split('\n'):
        line = line.strip()
        if line:
            try:
                labels.append(int(line))
            except ValueError:
                labels.append(0)
    return labels

def get_abstract_text(pmid):
    """Return the full abstract as a single string."""
    rec = load_one_abstract(pmid)
    if rec is None:
        return ''
    return ' '.join(rec['tokens'])

def get_gold_spans(pmid, field, split='train'):
    """
    Return the gold-standard extracted text for one field.
    Finds contiguous spans where label != 0 and joins the tokens.
    """
    rec = load_one_abstract(pmid)
    if rec is None:
        return []
    token_labels = load_ann_labels(pmid, field, split)
    if not token_labels:
        return []
    tokens = rec['tokens']
    spans  = []
    current_span = []
    for i, (tok, lab) in enumerate(zip(tokens, token_labels)):
        if lab != 0:
            current_span.append(tok)
        else:
            if current_span:
                spans.append(' '.join(current_span))
                current_span = []
    if current_span:
        spans.append(' '.join(current_span))
    return spans

# Loading 20 abstracts
N = 20
pmids = get_all_pmids()[:N]
print(f'Loaded {len(pmids)} abstracts for extraction')

#example to confirm that gold spans work
ex_pmid = pmids[0]
print(f'\nExample — {ex_pmid}')
for field in ['participants', 'interventions', 'outcomes']:
    spans = get_gold_spans(ex_pmid, field)
    print(f'  Gold {field}: {spans[:2]}')

Loaded 20 abstracts for extraction

Example — 10036953
  Gold participants: ['99', 'H. pylori infection']
  Gold interventions: ['H2 blockaders', 'ranitidine']
  Gold outcomes: ['Helicobacter pylori infections ]', 'Comparison']


## Approach A — End-to-end extraction

**Approach Steps** Given one prompt and will give model the full abstract and ask it to return all PICO fields at once.

**Expected:** High coverage, but lower precision.

In [ ]:
# Cell 7: Approach A — End-to-end QA extraction
QUESTIONS = {
    'participants': 'Who are the patients or participants in this trial?',
    'interventions': 'What treatment or intervention was given?',
    'outcomes':      'What outcomes or results were measured?'
}

def extract_end_to_end(abstract_text):
    result = {}
    for field, question in QUESTIONS.items():
        try:
            ans = qa_pipeline(
                question=question,
                context=abstract_text[:1000],
                max_answer_len=80
            )
            result[field] = ans['answer'] if ans['score'] > 0.05 else ''
        except Exception:
            result[field] = ''
    return result

# Run on ALL 20 abstracts to build results_a
print('Running Approach A on all abstracts...')
results_a = {}
for i, pmid in enumerate(pmids):
    text = get_abstract_text(pmid)
    results_a[pmid] = extract_end_to_end(text)
    print(f'  [{i+1}/{len(pmids)}] {pmid} done')

print('\nDone. results_a is ready.')
print('Example:', results_a[pmids[0]])

Running Approach A on all abstracts...
  [1/20] 10036953 done
  [2/20] 10037531 done
  [3/20] 10052279 done
  [4/20] 10071998 done
  [5/20] 10073522 done
  [6/20] 10075386 done
  [7/20] 10077140 done
  [8/20] 10078672 done
  [9/20] 10078673 done
  [10/20] 10080319 done
  [11/20] 10084579 done
  [12/20] 10089089 done
  [13/20] 10091821 done
  [14/20] 10093945 done
  [15/20] 10094243 done
  [16/20] 10097996 done
  [17/20] 10100592 done
  [18/20] 10148879 done
  [19/20] 10155556 done
  [20/20] 10172265 done

Done. results_a is ready.
Example: {'participants': '99', 'interventions': 'lansoprazole ( LPZ ) or ranitidine ( RNT )', 'outcomes': 'efficacy and safety'}


## Approach B — Decomposed 2-step extraction

**Step 1:** Classify each sentence as Participants / Interventions / Outcomes / Other.

**Step 2:** Now for sentences which are classified as relevant, we ask the model to extract the specific span for it.

**Expected:** Higher precision than A because each step is simpler and more focused.

In [ ]:
# Approach B — Decomposed 2-step
QUESTIONS = {
    'participants': 'Who are the patients or participants in this trial?',
    'interventions': 'What treatment or intervention was given?',
    'outcomes':      'What outcomes or results were measured?'
}

def classify_sentence(sentence):
    """
    Classify a sentence by asking which PICO question it best answers.
    We run all three questions and pick whichever gets the highest confidence.
    """
    best_field = 'other'
    best_score = 0.1

    for field, question in QUESTIONS.items():
        try:
            ans = qa_pipeline(
                question=question,
                context=sentence,
                max_answer_len=50
            )
            if ans['score'] > best_score:
                best_score = ans['score']
                best_field = field
        except Exception:
            continue

    return best_field


def extract_span_from_sentence(sentence, field):
    """Ask the specific question for this field against just this sentence."""
    try:
        ans = qa_pipeline(
            question=QUESTIONS[field],
            context=sentence,
            max_answer_len=80
        )
        return ans['answer'] if ans['score'] > 0.05 else ''
    except Exception:
        return ''


def extract_decomposed_2step(pmid):
    """
    Full 2-step pipeline for one abstract.
    Step 1: classify each sentence
    Step 2: extract span from relevant sentences
    """
    rec = load_one_abstract(pmid)
    if rec is None:
        return {'participants': '', 'interventions': '', 'outcomes': ''}

    tokens    = rec['tokens']
    sentences = rec['sentences']
    result    = {'participants': [], 'interventions': [], 'outcomes': []}

    for start, end in sentences:
        sentence = ' '.join(tokens[start:end]).strip()
        if not sentence:
            continue

        # Step 1: classifying this sentence
        field = classify_sentence(sentence)
        if field == 'other':
            continue

        # Step 2: extracting the specific span
        span = extract_span_from_sentence(sentence, field)
        if span:
            result[field].append(span)

    return {k: ' | '.join(v) for k, v in result.items()}


# Run it on all the 20 abstracts
print('Starting Approach B (decomposed 2-step)...')
print('This takes ~10 minutes — prints each abstract as it goes\n')

results_b = {}
for i, pmid in enumerate(pmids):
    print(f'  [{i+1}/{len(pmids)}] {pmid}', end=' ... ')
    results_b[pmid] = extract_decomposed_2step(pmid)
    print('done')

print('\nAll done!')
print('\nExample output for', pmids[0])
for field, val in results_b[pmids[0]].items():
    print(f'  {field}: {val[:100]}')

Starting Approach B (decomposed 2-step)...
This takes ~10 minutes — prints each abstract as it goes

  [1/20] 10036953 ... done
  [2/20] 10037531 ... done
  [3/20] 10052279 ... done
  [4/20] 10071998 ... done
  [5/20] 10073522 ... done
  [6/20] 10075386 ... done
  [7/20] 10077140 ... done
  [8/20] 10078672 ... done
  [9/20] 10078673 ... done
  [10/20] 10080319 ... done
  [11/20] 10084579 ... done
  [12/20] 10089089 ... done
  [13/20] 10091821 ... done
  [14/20] 10093945 ... done
  [15/20] 10094243 ... done
  [16/20] 10097996 ... done
  [17/20] 10100592 ... done
  [18/20] 10148879 ... done
  [19/20] 10155556 ... done
  [20/20] 10172265 ... done

All done!

Example output for 10036953
  participants: 99
  interventions: 
  outcomes: 


## Approach C — Decomposed 3-step extraction

**Approach Steps**: Steps are same as Approach B but we add a cleaning step that:
- Removes duplicate spans
- Removes spans which are too short (noise).
- Removes spans that are clearly wrong field.

**Expected:** Highest precision, but may lose some valid spans (lower coverage).

In [ ]:
# Approach C — Decomposed 3-step (adds cleaning on top of approch B)
def clean_spans(spans_text, field):
    if not spans_text:
        return ''
    spans = [s.strip() for s in spans_text.split('|')]
    cleaned = []
    seen = set()
    for span in spans:
        span = span.strip()
        if len(span.split()) < 2:
            continue
        key = span.lower()
        if key in seen:
            continue
        seen.add(key)
        if field == 'participants':
            person_words = ['patient', 'subject', 'participant', 'adult',
                           'child', 'women', 'men', 'age', 'year',
                           'male', 'female', 'children', 'infant']
            if not any(w in span.lower() for w in person_words):
                continue
        cleaned.append(span)
    return ' | '.join(cleaned)


def extract_decomposed_3step(pmid):
    raw = results_b[pmid]   #Reusing Apprach B results and cleaning them.
    return {
        field: clean_spans(raw[field], field)
        for field in ['participants', 'interventions', 'outcomes']
    }


print('Running Approach C (3-step cleaning)...')
results_c = {}
for i, pmid in enumerate(pmids):
    print(f'  [{i+1}/{len(pmids)}] {pmid}', end=' ... ')
    results_c[pmid] = extract_decomposed_3step(pmid)
    print('done')

print('\nAll done!')
print('\nComparison for', pmids[0])
print('  B (raw):    ', results_b[pmids[0]]['participants'])
print('  C (cleaned):', results_c[pmids[0]]['participants'])

Running Approach C (3-step cleaning)...
  [1/20] 10036953 ... done
  [2/20] 10037531 ... done
  [3/20] 10052279 ... done
  [4/20] 10071998 ... done
  [5/20] 10073522 ... done
  [6/20] 10075386 ... done
  [7/20] 10077140 ... done
  [8/20] 10078672 ... done
  [9/20] 10078673 ... done
  [10/20] 10080319 ... done
  [11/20] 10084579 ... done
  [12/20] 10089089 ... done
  [13/20] 10091821 ... done
  [14/20] 10093945 ... done
  [15/20] 10094243 ... done
  [16/20] 10097996 ... done
  [17/20] 10100592 ... done
  [18/20] 10148879 ... done
  [19/20] 10155556 ... done
  [20/20] 10172265 ... done

All done!

Comparison for 10036953
  B (raw):     99
  C (cleaned): 


In [ ]:
# Side-by-side comparison of all three approaches

print('=' * 80)
print('SIDE-BY-SIDE COMPARISON (first 3 abstracts)')
print('=' * 80)

for pmid in pmids[:3]:
    print(f'\nAbstract: {pmid}')
    print(f'Full text: {get_abstract_text(pmid)[:150]}...')
    print()

    for field in ['participants', 'interventions', 'outcomes']:
        gold  = get_gold_spans(pmid, field)
        gold_str = ' | '.join(gold[:2]) if gold else '(none)'
        a_str = results_a[pmid][field] or '(none)'
        b_str = results_b[pmid][field] or '(none)'
        c_str = results_c[pmid][field] or '(none)'

        print(f'  [{field.upper()}]')
        print(f'    Gold : {gold_str[:90]}')
        print(f'    A    : {a_str[:90]}')
        print(f'    B    : {b_str[:90]}')
        print(f'    C    : {c_str[:90]}')
    print('-' * 80)
    #This output gives us our qualitative findings

SIDE-BY-SIDE COMPARISON (first 3 abstracts)

Abstract: 10036953
Full text: [ Triple therapy regimens involving H2 blockaders for therapy of Helicobacter pylori infections ] . Comparison of ranitidine and lansoprazole in short...

  [PARTICIPANTS]
    Gold : 99 | H. pylori infection
    A    : 99
    B    : 99
    C    : (none)
  [INTERVENTIONS]
    Gold : H2 blockaders | ranitidine
    A    : lansoprazole ( LPZ ) or ranitidine ( RNT )
    B    : (none)
    C    : (none)
  [OUTCOMES]
    Gold : Helicobacter pylori infections ] | Comparison
    A    : efficacy and safety
    B    : (none)
    C    : (none)
--------------------------------------------------------------------------------

Abstract: 10037531
Full text: Xylitol for prevention of acute otitis media ....

  [PARTICIPANTS]
    Gold : acute otitis media .
    A    : (none)
    B    : (none)
    C    : (none)
  [INTERVENTIONS]
    Gold : Xylitol
    A    : (none)
    B    : (none)
    C    : (none)
  [OUTCOMES]
    Gold : (none)


In [16]:
#Saveing all results to a JSON file so we can load them later for evaluation without re-running the slow model calls
# Download all results to your computer
from google.colab import files
import json

# Save results locally in Colab first
with open('/content/extraction_results.json', 'w') as f:
    json.dump({
        'approach_a': results_a,
        'approach_b': results_b,
        'approach_c': results_c,
    }, f, indent=2)

# Download to your computer
files.download('/content/extraction_results.json')
print('Done.')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.
